In [1]:
#Imports
import psycopg2
import pandas as pd
from psycopg2 import sql


In [2]:
import os
from pathlib import Path

# Connection settings come from the environment (see .env.example at the repo root).
PG = dict(
    host=os.environ.get("POSTGRES_HOST", "localhost"),
    port=os.environ.get("POSTGRES_PORT", "5432"),
    dbname=os.environ.get("PEDIATRIC_DB", "pediatric"),
    user=os.environ.get("POSTGRES_USER", "postgres"),
    password=os.environ.get("POSTGRES_PASSWORD"),
)

#Koneksi PostgreSQL
try:
    conn = psycopg2.connect(**PG)
    print("Koneksi ke database pediatric berhasil!")
    cur = conn.cursor()
    print("Cursor berhasil dibuat!")
except psycopg2.Error as e:
    print("Gagal terhubung ke database:", e)

Koneksi ke database pediatric berhasil!
Cursor berhasil dibuat!


In [3]:
from pathlib import Path
DATA = Path.cwd().parent / "data"

#File paths
file_paths = {
    "biodata_pasien": str(DATA / "biodata_pasien.xlsx"),
    "pengkajian_pasien": str(DATA / "pengkajian_pasien.xlsx"),
    "laboratorium": str(DATA / "laboratorium.xlsx"),
    "konsultasi": str(DATA / "konsultasi.xlsx"),
    "farmasi": str(DATA / "farmasi.xlsx")
}

In [4]:
#Exract membaca file excel 
biodata_pasien_df = pd.read_excel(file_paths["biodata_pasien"])
pengkajian_pasien_df = pd.read_excel(file_paths["pengkajian_pasien"])
laboratorium_df = pd.read_excel(file_paths["laboratorium"])
konsultasi_df = pd.read_excel(file_paths["konsultasi"])
farmasi_df = pd.read_excel(file_paths["farmasi"])

In [5]:
#Display biodata_pasien
biodata_pasien_df.head()
#Display pengkajian_pasien
pengkajian_pasien_df.head()
#Display laborotarium
laboratorium_df.head()
#Display konsultasi
konsultasi_df.head()
#Display farmasi
farmasi_df.head()

,farmasi_id,pasien_id,nama_obat,dosis,frekuensi
0,91e8c384-af40-4284-afbc-1ef0f1d91e24,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,Paracetamol,3x sehari,6 hari
1,1cd437f2-4c9c-4a44-a46d-20f537a85058,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,Vitamin C,3x sehari,4 hari
2,e51f92c9-d0f8-413e-8e0c-c7e287eecdc4,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,Ibuprofen,1x sehari,6 hari
3,ba804ae4-38da-449d-80ba-a11266fcb7b2,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,Paracetamol,3x sehari,3 hari
4,e01225b6-672d-44f4-8f0a-880a8b07c0cc,8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e,Ibuprofen,1x sehari,7 hari


In [6]:
#Transform dan load dimensi biodata_pasien 
for index, row in biodata_pasien_df.iterrows():
    cur.execute("""
        INSERT INTO biodata_pasien (pasien_id, nama, umur, kelamin, alamat, no_tlp)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (pasien_id) DO NOTHING;
    """, (row['pasien_id'], row['nama'], row['umur'], row['kelamin'], row['alamat'], row['no_tlp']))

In [7]:
#Transform dan load dimensi pengkajian_pasien
for index, row in pengkajian_pasien_df.iterrows():
    cur.execute("""
        INSERT INTO pengkajian_pasien (pengkajian_id, pasien_id, tinggi, berat, tekanan, suhu, tinggi_bapak, tinggi_ibu)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (pengkajian_id) DO NOTHING;
    """, (row['pengkajian_id'], row['pasien_id'], row['tinggi'], row['berat'], row['tekanan'], 
          row['suhu'], row['tinggi_bapak'], row['tinggi_ibu']))

In [8]:
#Transform dan load dimensi laboratorium 
for index, row in laboratorium_df.iterrows():
    cur.execute("""
        INSERT INTO laboratorium (laboratorium_id, jenis_tes, hasil_tes, tanggal_tes)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (laboratorium_id) DO NOTHING;
    """, (row['laboratorium_id'], row['jenis_tes'], row['hasil_tes'], row['tanggal_tes']))

In [9]:
#Transform dan load dimensi konsultasi
for index, row in konsultasi_df.iterrows():
    cur.execute("""
        INSERT INTO konsultasi (konsultasi_id, nama_dokter, diagnosis, treatment, deskripsi)
        VALUES (%s, %s, %s, %s, %s)
        ON CONFLICT (konsultasi_id) DO NOTHING;
    """, (row['konsultasi_id'], row['nama_dokter'], row['diagnosis'], row['treatment'], row['deskripsi']))

In [10]:
#Transform dan load dimensi farmasi
for index, row in farmasi_df.iterrows():
    cur.execute("""
        INSERT INTO farmasi (farmasi_id, nama_obat, dosis, frekuensi)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (farmasi_id) DO NOTHING;
    """, (row['farmasi_id'], row['nama_obat'], row['dosis'], row['frekuensi']))


In [11]:
#Eksekusi
conn.commit()

In [12]:
#Display data
try:
    #Display biodata_pasien data
    cur.execute("SELECT * FROM biodata_pasien LIMIT 10;")
    biodata_pasien_data = cur.fetchall()
    print("Biodata Pasien:")
    for row in biodata_pasien_data:
        print(row)
    #Display pengkajian_pasien data
    cur.execute("SELECT * FROM pengkajian_pasien LIMIT 10;")
    pengkajian_pasien_data = cur.fetchall()
    print("\nPengkajian Pasien:")
    for row in pengkajian_pasien_data:
        print(row)
    #Display laboratorium data
    cur.execute("SELECT * FROM laboratorium LIMIT 10;")
    laboratorium_data = cur.fetchall()
    print("\nLaboratorium:")
    for row in laboratorium_data:
        print(row)
    #Display konsultasi data
    cur.execute("SELECT * FROM konsultasi LIMIT 10;")
    konsultasi_data = cur.fetchall()
    print("\nKonsultasi:")
    for row in konsultasi_data:
        print(row)
    #Display farmasi data
    cur.execute("SELECT * FROM farmasi LIMIT 10;")
    farmasi_data = cur.fetchall()
    print("\nFarmasi:")
    for row in farmasi_data:
        print(row)
    #Display fact_checkup data
    cur.execute("SELECT * FROM fact_checkup LIMIT 10;")
    fact_checkup_data = cur.fetchall()
    print("\nFact Checkup:")
    for row in fact_checkup_data:
        print(row)
except Exception as e:
    print("Error while fetching data:", e)

Biodata Pasien:
('8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e', 'John Doe', 10, 'Laki-laki', 'Jl. Mawar No.10, Jakarta', '81234567890', datetime.datetime(2026, 11, 17, 10, 51, 20, 472622), None, True)

Pengkajian Pasien:
('ec660332-32b0-4336-aaa4-00aed0679525', '8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e', Decimal('50.00'), Decimal('3.50'), '108', Decimal('36.09'), Decimal('170.00'), Decimal('160.00'), datetime.datetime(2026, 11, 17, 10, 51, 20, 472622), None, True)
('80873108-c99e-441e-8850-7dc8e618bdf9', '8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e', Decimal('52.00'), Decimal('4.70'), '103', Decimal('36.54'), Decimal('170.00'), Decimal('160.00'), datetime.datetime(2026, 11, 17, 10, 51, 20, 472622), None, True)
('861dd05b-8c9d-4ff6-882d-98cfb0f6de10', '8ddfe7ed-f7e8-4d4b-a31f-97a298d5d82e', Decimal('54.00'), Decimal('5.90'), '96', Decimal('36.58'), Decimal('170.00'), Decimal('160.00'), datetime.datetime(2026, 11, 17, 10, 51, 20, 472622), None, True)
('e6949615-61c7-4444-8d6e-f2d545b8888c', '8ddfe7ed-f7e8